<a href="https://colab.research.google.com/github/Vivekshrotriya1/23_March/blob/main/Question_1_Build_an_End_to_End_RAG_%2B_Agent_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Question 1: Build an End-to-End RAG + Agent System (25 Marks)
 # Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:

Course policies (refunds, deadlines):
- Lecture content (PDF notes)
- Assignment deadlines
- Their enrollment status

The company wants a single intelligent assistant that:

- Answers questions using internal documents (PDFs, FAQs)
- Fetches student-specific data (like enrollment or progress) using tools/APIs
- Avoids hallucination and gives reliable answers

💻 Task
 - Design and implement a working prototype (pseudo-code or real code) for this system.

Your solution must include:

✅ 1. RAG Pipeline
- How you will ingest and preprocess documents
- Chunking strategy (with justification)
- Embedding + vector store choice
- Retrieval logic
- How context is injected into the LLM

✅ 2. Agent Integration
- Design an agent that decides:
- When to use RAG
- When to call a tool (e.g., get_student_status(student_id))
- Show how tools are defined and used

✅ 3. End-to-End Flow-
Write code or structured pseudo-code showing:

- Input query
- Retrieval step
- Tool calling (if needed)
- Final answer generation

✅ 4. Reliability Improvements -Show at least 2 techniques in code/design to:

- Reduce hallucination
- Improve answer quality

🎯 Expected Outcome:
A working architecture/code that demonstrates:

- RAG + Agent working together
- Decision-making capability
- Real-world practicality

In [22]:

!pip install groq sentence-transformers faiss-cpu numpy

# -------- IMPORTS ----------
from groq import Groq
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import re
from google.colab import userdata

# -------- GROQ SETUP (SECURE) ----------
api_key = userdata.get('VivekApi')

if api_key is None:
    raise ValueError("❌ API key not found. Add it in Colab Secrets.")

client = Groq(api_key=api_key)

# -------- MODEL ----------
MODEL_NAME = "llama-3.1-8b-instant"

# -------- RAW DATA ----------
documents = [
    "Refund policy: Students can request refund within 7 days of enrollment.",
    "Assignment deadlines are strict and late submissions are not accepted.",
    "Lecture 1 covers Machine Learning basics and supervised learning.",
    "Course deadline: Complete within 3 months from enrollment."
]

student_db = {
    "101": {"name": "Vivek", "enrolled": True, "progress": "60%"},
    "102": {"name": "Rahul", "enrolled": False, "progress": "0%"}
}

# -------- CHUNKING ----------
def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))

# -------- EMBEDDING MODEL ----------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# -------- CREATE EMBEDDINGS ----------
chunk_embeddings = embed_model.encode(all_chunks)

# -------- FAISS INDEX ----------
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

# -------- LLM CALL ----------
def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a helpful ed-tech assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ LLM Error: {str(e)}"

# -------- RETRIEVAL ----------
def retrieve(query, k=2):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    results = [all_chunks[i] for i in indices[0]]
    return results

# -------- RAG ----------
def rag_answer(query):
    retrieved_chunks = retrieve(query)

    if not retrieved_chunks:
        return "I don't know."

    context = "\n".join(retrieved_chunks)

    prompt = f"""
    STRICT RULE:
    Answer ONLY from the context below.
    If answer not found, say "I don't know".

    Context:
    {context}

    Question:
    {query}
    """

    return call_llm(prompt)

# -------- TOOL ----------
def get_student_status(student_id):
    data = student_db.get(student_id)
    if not data:
        return "Student not found"

    return f"""
    Name: {data['name']}
    Enrolled: {data['enrolled']}
    Progress: {data['progress']}
    """

# -------- HELPER ----------
def extract_id(query):
    match = re.search(r"\d+", query)
    return match.group() if match else None

# -------- AGENT ----------
def agent(query):

    # Decide: Tool or RAG
    if any(word in query.lower() for word in ["progress", "status", "enrollment"]):

        student_id = extract_id(query)

        if not student_id:
            return "Please provide your student ID."

        tool_output = get_student_status(student_id)

        prompt = f"""
        User Question: {query}
        Tool Output: {tool_output}

        Generate a helpful response.
        """

        return call_llm(prompt)

    else:
        return rag_answer(query)

# -------- MAIN ----------
def run_system():
    print("🎓 EdTech AI Assistant (Pure RAG + Agent)")
    print("Type 'exit' to quit\n")

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            break

        response = agent(query)
        print("AI:", response)
        print("-" * 50)

# -------- RUN ----------
if __name__ == "__main__":
    run_system()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎓 EdTech AI Assistant (Pure RAG + Agent)
Type 'exit' to quit

User: what is refund policy ?
AI: Students can request a refund within 7 days of enrollment.
--------------------------------------------------
User: exit
